# 03 — Which Item 7 should FinBERT read?

Here we decide which source to use for item 7 of the report.

Plan:
1. Read some disagreeing cases by eye
2. How often does each source look wrong, using simple text markers?

In [1]:
import gzip
import json
import re
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..")
PROCESSED = ROOT / "data" / "processed"
kogan_dir = ROOT / "data" / "raw" / "kogan"
edgar_dir = PROCESSED / "edgar_matched"

df = pd.read_csv(PROCESSED / "kogan_edgar_merged.csv")
print(df.shape)
df.head(3)

(30465, 20)


,key,date,name,cik,logvol_before,logvol_after,year,two_in_year,kogan_item7_words,edgar_item1_words,edgar_item1A_words,edgar_item7_words,name_found,matched,has_label,has_item1,has_item1A,item7_ratio,item7_lengths_agree,usable
0,000794107-10-K-19960401,1996-04-01,ACC CORP,783233,-3.665589,-3.153746,1996,False,6895,22055.0,0.0,52.0,True,True,True,True,False,0.007542,False,True
1,00087B101-10-K-19960529,1996-05-29,ACR GROUP INC,711307,-2.261435,-2.568802,1996,False,2607,2087.0,0.0,2614.0,True,True,True,True,False,1.002685,True,True
2,000886309-10-K-19960105,1996-01-05,ADC TELECOMMUNICATIONS INC,61478,-3.245127,-3.400733,1996,False,1900,6907.0,0.0,1471.0,True,True,True,True,False,0.774211,True,True


## 1. Helpers to fetch both texts for a handful of reports

Reading all 28,000 texts into memory would be too much, so these functions pull out only the reports we ask for (a *set* of `(cik, year)` pairs)

In [2]:
def get_texts(rows):
    """Return {(cik, year): (kogan_text, edgar_text)} for the given rows of df."""
    out = {}
    for y, group in rows.groupby("year"):
        wanted_keys = {r.key: (r.cik, r.year) for r in group.itertuples()}
        with tarfile.open(kogan_dir / f"{y}.tok.tgz") as tar:
            for m in tar:
                if m.isfile():
                    key = Path(m.name).name.split(".")[0]
                    if key in wanted_keys:
                        out[wanted_keys[key]] = [tar.extractfile(m).read().decode("utf-8", "replace"), None]
        wanted = set(c for c, _ in wanted_keys.values())
        with gzip.open(edgar_dir / f"{y}.jsonl.gz", "rt") as f:
            for line in f:
                rec = json.loads(line)
                if rec["cik"] in wanted:
                    out[(rec["cik"], rec["year"])][1] = rec["section_7"]
    return out


import textwrap

def _block(label, words, n):
    """Wrap the first and last n words of one text into short lines."""
    def wrap(ws):
        return textwrap.fill(" ".join(ws), width=88, initial_indent="      ", subsequent_indent="      ")
    print(f"  {label}  ({len(words):,} words)")
    print("    starts:")
    print(wrap(words[:n]))
    print("    ends:")
    print(wrap(words[-n:]))


def show(rows, n=40):
    """Print the first and last n words of both versions for each row, one block per report."""
    texts = get_texts(rows)
    for i, r in enumerate(rows.itertuples(), 1):
        k, e = texts[(r.cik, r.year)]
        print("=" * 92)
        print(f"{i}. {r.name} ({r.year})   EDGAR / Kogan length = {len(e.split()) / len(k.split()):.2f}")
        print("-" * 92)
        _block("KOGAN", k.split(), n)
        print()
        _block("EDGAR", e.split(), n)
        print()

## 2. Inspect cases where EDGAR is much *shorter* than Kogan (ratio < 0.5)

In [3]:
short = df[df["matched"] & (df["edgar_item7_words"] >= 100) & (df["item7_ratio"] < 0.5)]
print(len(short), "reports; reading 8 at random")
show(short.sample(8, random_state=1))

1871 reports; reading 8 at random


1. DORCHESTER MINERALS LP (2005)   EDGAR / Kogan length = 0.40
--------------------------------------------------------------------------------------------
  KOGAN  (9,781 words)
    starts:
      item # management s discussion and analysis of financial condition and results of
      operations basis of presentation in the combination completed on january # # and
      accounted for as a purchase dorchester hugoton was designated as the accounting
      acquiror prior
    ends:
      incurring any debt other than trade debt therefore we do not expect interest rate
      risk to be material to us we do not anticipate engaging in transactions in foreign
      currencies which could expose us to foreign currency related market risk

  EDGAR  (3,914 words)
    starts:
      ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF
      OPERATIONS Basis of Presentation In the combination completed on January 31, 2003
      and accounted for as a purchase, Dorcheste

Things to look for in each case:
- Does the **EDGAR end** stop at a sentence like "see the Annual Report to Shareholders" (a genuine cut-off)?
- Does the **Kogan end** drift into financial statements ("report of independent auditors", "consolidated balance sheets")? That would mean Kogan's version is too long because it swallowed Item 8.

**My reading of the 8 cases where EDGAR is shorter (ratio < 0.5)**

1. **Dorchester Minerals (2005):** both start at the same place ("basis of presentation"). EDGAR ends cleanly at the "ITEM 7A" heading. Kogan is 2.5x longer and ends in the market-risk text, so it includes Item 7A and more.
2. **Exactech (2005):** EDGAR runs from the overview to the "ITEM 7A" heading. Kogan starts at "liquidity and capital resources", not at the beginning of the section, and ends on "table of contents", which is page furniture.
3. **Cutera (2005):** same start in both. EDGAR ends at "ITEM 7A". Kogan carries on into foreign-currency and derivatives risk, which is Item 7A content.
4. **PNM Resources (2002):** Kogan is 35,756 words against EDGAR's 9,991, and starts at "other issues facing the company", partway through. EDGAR starts at the proper opening paragraph. A utility's Item 7 can be long, but 3.6x is more than one section.
5. **Remington Oil & Gas (2004):** same start. Kogan ends on the derivatives disclosure that EDGAR only points to ("see Item 7A"), so Kogan includes 7A.
6. **Broadcom (2005):** same start. Kogan's ending is about strategic investments (market risk). EDGAR stops at the "Item 7A" heading.
7. **Kaiser Ventures (2000):** both end on the same sentence (the tax "annual limitation"), but Kogan is 4x longer and starts mid-discussion ("for additional information in addition to developing the kaiser commerce center"). EDGAR starts at "Section 1: Operating Results Summary". Kogan has extra material at the front that I can't explain.
8. **HighwayMaster (1999):** Kogan starts at "limitations due to debt", mid-section. It ends with the Item 7A heading ("not applicable"). EDGAR starts at "GENERAL" and ends at "ITEM 7A".

**What this shows**
- In all 8, EDGAR is one clean block that runs from the "Item 7" heading to the next heading.
- In all 8, Kogan is 2 to 4.7x longer. At least 5 of the 8 include Item 7A text, and 4 of the 8 start partway through the section.
- Kogan is not cut short here. It has **more** text than EDGAR, but the extra is partly Item 7A and partly something I can't pin down.
- Neither looks like a failed extraction. This group is a difference in what each source counts as Item 7, so using EDGAR here loses little.

## 3. Inspect cases where EDGAR is much *longer* than Kogan (ratio > 2)

In [4]:
long_ = df[df["matched"] & (df["item7_ratio"] > 2)]
print(len(long_), "reports; reading 8 at random")
show(long_.sample(8, random_state=1))

668 reports; reading 8 at random


1. BLOCKBUSTER INC (2002)   EDGAR / Kogan length = 2.74
--------------------------------------------------------------------------------------------
  KOGAN  (10,928 words)
    starts:
      item # managements discussion and analysis of financial condition and results of
      operations the following discussion and analysis should be read in conjunction
      with the consolidated financial statements and related notes appearing elsewhere
      in this document general blockbuster inc is
    ends:
      a material effect on our consolidated financial statements item # a quantitative
      and qualitative disclosures about market risk response to this item is included in
      item # managements discussion and analysis of financial condition and results of
      operationsmarket risk #

  EDGAR  (29,961 words)
    starts:
      Item 7. Management’s Discussion and Analysis of Financial Condition and Results of
      Operations (Tabular dollars in millions) The following discussion and a

1. **Blockbuster (2002):** Kogan looks complete (it ends at the market-risk cross-reference). **EDGAR overshoots:** it ends on an exhibit list ("incorporated herein by reference... Filed herewith"), so it has run well past Item 7 (29,961 words). This is a genuine EDGAR error.
2. **Vista Gold (2003):** Kogan ends with the small-business Item 7A statement and has 1,952 words. EDGAR is 5,275 words and ends at the "ITEM 7A" heading. Kogan seems to have missed part of the body.
3. **Newmont Gold (1997):** **Kogan is cut off mid-sentence** ("...as discussed in note # of"). It stopped at the cross-reference. EDGAR has the full section and ends at "ITEM 7A".
4. **Woodward Governor (2006):** **Kogan is cut off** ("information is included in the notes to consolidated financial statements in"). EDGAR is complete (9,267 words).
5. **American States Water (2004):** **Kogan is cut off** ("included in part ii"). Its start also has the heading glued to the next words ("operationrisk factors"). EDGAR is complete (22,818 words).
6. **Amazon (2005):** **Kogan is only the forward-looking statements intro** (1,390 words, ending "for additional information... refer to"). EDGAR has the whole 10,403-word section. This is the clearest Kogan failure.
7. **DSP Group (2006):** **Kogan is cut off** ("included in") at a cross-reference. EDGAR is complete.
8. **CIDCO (1997):** Kogan ends on a full sentence, so no obvious cut-off. It is 2.9x shorter than EDGAR, which ends at "ITEM 7A", so some body text may be missing. I can't tell from the ends alone.

**What this shows**
- In 5 of 8 cases (Newmont, Woodward, American States Water, Amazon, DSP) Kogan stops mid-sentence at a cross-reference. That is the failure the 2009 paper's extraction is prone to.
- EDGAR is a clean, complete section in 7 of 8. The exception is Blockbuster, where it overshoots into exhibits.
- Kogan's text is also lowercased, with numbers turned into `#`, apostrophes split ("management s") and hyphens removed ("longterm", "forwardlooking").

## 4. Some simple markers for a bad extraction

Eyeballing 16 cases can't show how common each mistake is, so we search every text for two warning signs.

- spill: contains "report of independent" (an auditor's report, which belongs to Item 8, so the text has run past the end of Item 7)
- by_reference (EDGAR only): short text that says the content is *incorporated by reference* or points to an *Annual Report to Shareholders* (the real text is in an exhibit, so the section is legitimately nearly empty)

We go through every report once and keep only the yes/no flags.

In [5]:
SPILL = re.compile(r"report of independent")
BYREF = re.compile(r"incorporated (herein )?by reference|annual report to (share|stock)holders")

flags_file = PROCESSED / "item7_markers.csv"
if flags_file.exists():
    markers = pd.read_csv(flags_file)
else:
    rows = []
    for y in range(1996, 2007):
        kogan_spill = {}
        with tarfile.open(kogan_dir / f"{y}.tok.tgz") as tar:
            for m in tar:
                if m.isfile():
                    kogan_spill[Path(m.name).name.split(".")[0]] = bool(SPILL.search(tar.extractfile(m).read().decode("utf-8", "replace")))
        edgar_flags = {}
        with gzip.open(edgar_dir / f"{y}.jsonl.gz", "rt") as f:
            for line in f:
                rec = json.loads(line)
                t = rec["section_7"].lower()
                edgar_flags[(rec["cik"], rec["year"])] = (bool(SPILL.search(t)), bool(BYREF.search(t[:3000])))
        for r in df[df["year"] == y].itertuples():
            es, eb = edgar_flags.get((r.cik, r.year), (np.nan, np.nan))
            rows.append({"key": r.key, "kogan_spill": kogan_spill.get(r.key, np.nan), "edgar_spill": es, "edgar_byref": eb})
    markers = pd.DataFrame(rows)
    markers.to_csv(flags_file, index=False)

d = df.merge(markers, on="key")
d = d[d["matched"]].copy()
d["group"] = pd.cut(d["item7_ratio"].fillna(0), [-1, 0.5, 2, np.inf], labels=["EDGAR < 0.5x Kogan", "agree (0.5-2x)", "EDGAR > 2x Kogan"]).astype(str)
d.loc[d["edgar_item7_words"] < 100, "group"] = "EDGAR missing (<100 words)"
table = d.groupby("group", observed=True).agg(reports=("key", "size"),
                                              kogan_spill=("kogan_spill", "mean"),
                                              edgar_spill=("edgar_spill", "mean"),
                                              edgar_byref=("edgar_byref", "mean")).round(3)
table

,reports,kogan_spill,edgar_spill,edgar_byref
group,,,,
EDGAR < 0.5x Kogan,1871,0.019,0.002138,0.042758
EDGAR > 2x Kogan,668,0.006,0.049401,0.031437
EDGAR missing (<100 words),727,0.010,0.0,0.313618
agree (0.5-2x),25020,0.007,0.002998,0.023861


- If kogan_spill is high where EDGAR is shorter, Kogan's text is too long (it swallowed other sections), so EDGAR is right.
- If edgar_byref is high where EDGAR is shorter, EDGAR is short because the report really does leave Item 7 to an exhibit, and Kogan (which got the exhibit text) is the fuller one.

Findings:
- EDGAR missing (727 reports): 31% of these say "incorporated by reference" or point to an Annual Report to Shareholders, against about 2% for normal reports. These are real gaps, not extraction errors, and Kogan holds the text. This supports falling back to Kogan here.
- EDGAR shorter (1,871): Kogan's auditor-report marker fires on only 1.9% and EDGAR's by-reference marker on 4.3%. Neither mistake explains why these reports differ, which fits what I read - they differ mostly because Kogan includes Item 7A and other extra text.
- EDGAR longer (668): EDGAR's spill marker fires on 4.9%, against 0.3% for agreeing reports, so EDGAR does overrun into later sections for about 1 in 20 of these (like Blockbuster). Kogan's spill rate is only 0.6%, because its problem is being cut short, which this marker can't see.
- Agree (25,020): every marker is at 1% or less, so these are clean.

## 5. Chosen method

FinBERT reads EDGAR's Item 7. If EDGAR has fewer than 100 words, it reads Kogan's instead.

- EDGAR is the better text. Where the two disagree a lot, Kogan is often cut off at a cross-reference (5 of 8 cases), while EDGAR is a clean section and is ordinary prose, which is what FinBERT expects. Item 1 also comes from EDGAR, so both sections come from one extractor.
- Some reports leave Item 7 to an exhibit ("incorporated by reference"), so EDGAR's section is empty but Kogan has the text.
- limitations: the 100-word cut-off is a simple rule


In [6]:
# EDGAR's Item 7, unless it has fewer than 100 words; then Kogan's
out = df[["key", "cik", "year", "usable"]].copy()
out["item7_source"] = np.where(df["edgar_item7_words"] >= 100, "edgar", "kogan")
out.loc[~out["usable"], "item7_source"] = np.nan

print(out.groupby("year")["item7_source"].value_counts().unstack().fillna(0).astype(int))
out.to_csv(PROCESSED / "item7_source.csv", index=False)

item7_source  edgar  kogan
year                      
1996           1190     68
1997           1904     72
1998           2086     56
1999           2160     60
2000           2108     43
2001           2265     34
2002           2537     50
2003           3154     63
2004           3116     52
2005           3072     49
2006           2946     57
